# Swarm-MeZO — Day 4: reputational consensus on RoBERTa+SST-2 (Colab)

Запускает `scripts/run_reputation.py` на удалённой GPU в Colab.

**Что делает прогон:** N=8 агентов, non-IID Dirichlet(α=0.5) split, K=100 локальных MeZO-шагов между раундами консенсуса, 5000 шагов всего; sweep по `β ∈ {0, 0.1, 0.5, 1, 10}` с репутационной матрицей `W_ij = r_j/Σr`.

**Время:** ≈1.5ч на конфиг в bf16 на L4/A100, ≈3ч на T4 (нет bf16 tensor cores → fp32 fallback). Итого 5 конфигов = 7–15 часов.

**Перед запуском:** Runtime → Change runtime type → GPU (желательно A100/L4/V100, T4 пойдёт но медленно).

## 1. Setup: клонирование и зависимости

In [1]:
!nvidia-smi | head -20

Tue May 19 21:29:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Клонируем репозиторий (публичный, доступ без токена).
import os, subprocess
REPO_DIR = '/content/swarm-mezo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/olegkeatzin/swarm-mezo.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull
%cd $REPO_DIR

Cloning into '/content/swarm-mezo'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 158 (delta 57), reused 138 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.98 MiB | 19.16 MiB/s, done.
Resolving deltas: 100% (57/57), done.
/content/swarm-mezo


In [3]:
# Зависимости — на Colab используем pip, а не uv (уже есть torch с правильной CUDA).
# Если по какой-то причине нужна свежая версия torch — раскомментировать строку ниже.
# !pip install -q --upgrade torch
!pip install -q 'transformers>=4.40' 'datasets>=2.20' tqdm matplotlib pandas
import torch
print(f'torch {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap}  (bf16 fast on cap >= (8,0))')

torch 2.10.0+cu128, cuda available: True
gpu: Tesla T4
compute capability: (7, 5)  (bf16 fast on cap >= (8,0))


## 2. Прогрев кеша: модель + датасет

Скачивает RoBERTa-base и SST-2 в `~/.cache/huggingface` чтобы во время прогона не было сетевых задержек.

In [4]:
from datasets import load_dataset            # ВАЖНО: импортируется ДО torch в скриптах
from transformers import AutoModelForMaskedLM, AutoTokenizer

_ = load_dataset('glue', 'sst2')
_ = AutoTokenizer.from_pretrained('roberta-base')
_ = AutoModelForMaskedLM.from_pretrained('roberta-base')
print('cache OK')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cache OK


## 3. Санитарный smoke-test (≈1 минута)

10 шагов × N=2 чтобы убедиться что vmap + bf16 + reputation-путь не падают на этой GPU.

In [5]:
!python scripts/smoke_test_reputation.py 2>&1 | tail -40

device: cuda
Loading model...
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1066.42it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Running 10-step reputation smoke test (N=2, K=5, β=10)...
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1130.58it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical 

## 4. Главный прогон

Скрипт идемпотентный — пишет результаты инкрементально в `outputs/day5_reputation.json` после каждого `β`. Если Colab разорвёт сессию посередине, перезапуск этой ячейки продолжит с недосчитанного.

Запускаем через `tee`, чтобы был и лог на диске, и output в Colab UI.

In [ ]:
!mkdir -p outputs && python -u scripts/run_reputation.py 2>&1 | tee -a outputs/day5_reputation.log

device: cuda
gpu:    Tesla T4

Non-IID per-agent class balance (neg, pos):
  agent 0: neg= 27, pos= 24
  agent 1: neg=  2, pos= 14
  agent 2: neg=184, pos= 13
  agent 3: neg= 78, pos= 64
  agent 4: neg=  5, pos= 25
  agent 5: neg= 18, pos=  3
  agent 6: neg=  4, pos=415
  agent 7: neg=122, pos=  2
reputation probe batch: 32 samples (neg=18, pos=14)

Loading model weights...
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1058.17it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForMaskedLM LOAD REPORT from: roberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Weights cached.

Day-5 reputation sweep (N=8, K=100, steps=5000)

--- beta0.0  (β=0.0, γ_r=1.0) ---
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1153.73i

## 5. Быстрый превью результатов

Финальная val accuracy и эволюция репутаций для каждого `β` — диагностика каскада.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

d = json.loads(open('outputs/day5_reputation.json').read())
runs = dict(sorted(d['runs'].items(), key=lambda kv: kv[1]['beta']))
print(f'{len(runs)} configs in {list(runs.keys())}')
print()
print(f"{'β':>6} | {'final val_acc':>14} | {'final loss':>11} | {'max rep share':>14}")
for name, h in runs.items():
    reps = h.get('reputations') or []
    if reps:
        last = np.asarray(reps[-1])
        max_share = float(last.max() / last.sum())
    else:
        max_share = float('nan')
    print(f"{h['beta']:>6} | {h['eval_acc'][-1]:>14.4f} | {h['eval_loss'][-1]:>11.4f} | {max_share:>14.3f}")

In [ ]:
# Кривые val accuracy на лету (полный анализ — в notebooks/05_day4_reputation.ipynb)
fig, ax = plt.subplots(figsize=(8, 5), dpi=110)
cmap = plt.get_cmap('viridis')
for i, (name, h) in enumerate(runs.items()):
    color = cmap(i / max(len(runs) - 1, 1))
    ax.plot(h['eval_step'], h['eval_acc'], marker='o', markersize=3, color=color,
            label=f"β = {h['beta']}")
ax.set_xlabel('per-agent step')
ax.set_ylabel('SST-2 val accuracy')
ax.set_title('Day 4: reputational MeZO on RoBERTa+SST-2')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 6. Скачивание результатов

Заберите JSON + лог себе локально, чтобы прогнать полную визуализацию в `notebooks/05_day4_reputation.ipynb`.

In [ ]:
from google.colab import files
files.download('outputs/day5_reputation.json')
files.download('outputs/day5_reputation.log')

**Альтернатива** — закоммитить и запушить результаты прямо из Colab (потребуется GitHub PAT в `userdata`):

```python
from google.colab import userdata
token = userdata.get('GITHUB_PAT')
!cd /content/swarm-mezo && \
  git config user.email 'colab@local' && \
  git config user.name 'Colab Runner' && \
  git add outputs/day5_reputation.json outputs/day5_reputation.log && \
  git commit -m 'Day 4: reputational β sweep on RoBERTa+SST-2' && \
  git push https://$token@github.com/olegkeatzin/swarm-mezo.git main
```